# Chaining jobs (official Python client only)

Same story as [`chaining_jobs.ipynb`](./chaining_jobs.ipynb) — register a spreadsheet as a Model, run `@istari:extract`, chain a second job off `workbook.xlsx`, and inspect lineage — but **without** [`istari_labs_helpers`](../istari-labs-helpers). Everything below uses the v2 **`istari_digital_client.Client`** API directly.

### Prerequisites

- **`istari-digital-client`** and **`python-dotenv`** (the [`istari-labs-helpers`](../istari-labs-helpers) tutorial venv from `uv sync --project istari-labs-helpers --extra experiment` already includes them; you can import `Client` only and ignore `istari_labs_helpers`).
- The same `.env` as the other notebook: `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN` in `samples/.env`.
- **Open Spreadsheet** + `@istari:extract` access for your user.
- `Group3-UAS-Requirements.xlsx` next to this notebook.

### Install (minimal environment)

```bash
pip install istari-digital-client python-dotenv
```


## 1 · Connect and verify

We build `Configuration` from the same env vars as the official docs, construct `Client`, then call `readiness_check()` (or you can hit any lightweight API) to prove the token works.


In [ ]:
import json
import os
import tempfile
import time
from pathlib import Path

import dotenv
from istari_digital_client import Client, Configuration, JobStatusName
from istari_digital_client.v2.models.new_source import NewSource

dotenv.load_dotenv()
_registry_url = os.environ.get("ISTARI_REGISTRY_URL")
_token = os.environ.get("ISTARI_PERSONAL_ACCESS_TOKEN")
if not _registry_url or not _token:
    raise RuntimeError("Set ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN (e.g. in samples/.env)")

client = Client(Configuration(registry_url=_registry_url, registry_auth_token=_token))

report = client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"
print("Connected:", _registry_url)


## 2 · Register the spreadsheet as a Model

`add_model` uploads the file and creates the Model resource + first revision.


In [ ]:
XLSX_PATH = Path.cwd() / "Group3-UAS-Requirements.xlsx"
EXTERNAL_ID = "sdk-tutorial-uas-requirements"
DISPLAY_NAME = "Group3-UAS-Requirements (tutorial sdk)"

if not XLSX_PATH.exists():
    raise FileNotFoundError(f"Expected {XLSX_PATH.resolve()} — run from samples/")

model = client.add_model(
    XLSX_PATH,
    external_identifier=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded new model {model.id}")
print("Name:", getattr(model, "name", None), "file:", model.file.id if model.file else None)


## 3 · Run the first extraction job

`add_job` needs `model_id`, `function`, and `tool_name`. Poll with `get_job` until status is **Completed** or **Failed** (see `JobStatusName`).


In [ ]:
def wait_job(job_id: str, *, timeout: int = 600, poll_interval: int = 5, on_poll=None):
    start = time.time()
    while True:
        job = client.get_job(job_id)
        if on_poll is not None:
            st = job.status.name.value if job.status else "unknown"
            try:
                on_poll(st, job.id)
            except Exception as exc:
                print("[wait] on_poll raised:", exc)
        name = job.status.name
        if name == JobStatusName.PENDING:
            msg = job.status.message
            if msg and "None agent" in msg:
                raise RuntimeError(f"No agent available: {msg}")
        if name in (JobStatusName.COMPLETED, JobStatusName.FAILED):
            return job
        if time.time() - start > timeout:
            return job
        time.sleep(poll_interval)


def require_completed(job):
    if job.status.name != JobStatusName.COMPLETED:
        raise RuntimeError(f"Job {job.id} ended with {job.status.name!s}")
    return job


FUNCTION = "@istari:extract"
TOOL = "open_spreadsheet"

job1_raw = client.add_job(model.id, FUNCTION, tool_name=TOOL)
print(f"Submitted job {job1_raw.id}; polling...")

job1 = wait_job(
    job1_raw.id,
    timeout=600,
    on_poll=lambda st, jid: print(f"  [{st}] id={jid}"),
)
job1 = require_completed(job1)
print(f"\nJob 1 finished: {job1.status.name}")


## 4 · Inspect the products

Completed jobs expose output metadata on the job output **`File`**: use **`job.file.revision`** (latest revision) and its **`products`** list. Each **`Product`** is client-wired after `get_job`, so **`p.revision`** loads the pinned output `FileRevision` (same as `get_revision`, but idiomatic).


In [ ]:
def iter_job_products(job):
    job = client.get_job(job.id)
    if not job.file or not job.file.revisions:
        return
    rev = job.file.revision
    for p in rev.products or []:
        if p.resource_type and p.resource_id and p.revision_id:
            yield p


def print_job_products(job, label="products"):
    items = list(iter_job_products(job))
    print(f"Job wrote {len(items)} {label}:\n")
    for p in items:
        rrev = p.revision
        if rrev is None:
            print(f"  - {p.resource_type:10s}  (could not load revision {p.revision_id})")
            continue
        print(
            f"  - {p.resource_type:10s}  name={rrev.name!r:30s}  file={rrev.file_id}  rev={p.revision_id}"
        )
    return items


products_1_meta = print_job_products(job1)


## 5 · Pick a product and download it

`read_contents` streams bytes for a revision (`content_token`). `named_cells.json` is JSON text.


In [ ]:
def find_product_by_filename(job, filename: str):
    for p in iter_job_products(job):
        rrev = p.revision
        if rrev is not None and rrev.name == filename:
            return p, rrev
    return None, None


named_cells_p, named_cells_rev = find_product_by_filename(job1, "named_cells.json")
assert named_cells_rev is not None, "named_cells.json not produced"

raw = client.read_contents(token=named_cells_rev.content_token)
data = json.loads(raw.decode("utf-8"))
print(f"named_cells.json has {len(data)} named ranges")
print("First 3 keys:", list(data)[:3])

out_path = Path.cwd() / "outputs.json"
out_path.write_bytes(raw)
print(f"Downloaded to: {out_path}")


## 6 · Chain a second job

Jobs only accept a **Model** id. `istari_labs_helpers` calls `add_model` with a `NewSource` pointing at the artifact revision (promotion). Reproduce that here: download the pinned `workbook.xlsx` revision, re-upload as a new Model linked via `promoted_from`, then `add_job` on that Model.


In [ ]:
workbook_p, workbook_rev = find_product_by_filename(job1, "workbook.xlsx")
assert workbook_rev is not None, "Job 1 should have produced workbook.xlsx"
print("Chaining off revision:", workbook_rev.id, workbook_rev.name)


def promote_revision_to_model(revision, *, external_identifier=None):
    """Mirror istari_labs_helpers auto-promotion: new Model whose source is this revision."""
    content = client.read_contents(token=revision.content_token)
    upload_name = revision.name or f"promote-{revision.id}.xlsx"
    tmp_dir = tempfile.mkdtemp(prefix="istari_promote_")
    tmp_path = os.path.join(tmp_dir, upload_name)
    try:
        with open(tmp_path, "wb") as f:
            f.write(content)
        promoted = client.add_model(
            path=tmp_path,
            display_name=Path(upload_name).stem,
            external_identifier=external_identifier,
            sources=[
                NewSource(revision_id=revision.id, relationship_identifier="promoted_from")
            ],
        )
        return promoted
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
        if os.path.isdir(tmp_dir):
            os.rmdir(tmp_dir)


promoted = promote_revision_to_model(workbook_rev)
print("Promoted model:", promoted.id)

job2_raw = client.add_job(promoted.id, FUNCTION, tool_name=TOOL)
job2 = wait_job(
    job2_raw.id,
    timeout=600,
    on_poll=lambda st, jid: print(f"  [{st}] id={jid}"),
)
job2 = require_completed(job2)
print(f"\nJob 2 finished: {job2.status.name}")

products_2_meta = print_job_products(job2, "products")


## 7 · Trace lineage (SDK walk)

`istari_labs_helpers` folds the raw graph into a compact tree. With the client alone we still have full provenance: walk `get_revision`, print each revision and its `sources`, and recurse. This is **informationally equivalent** to the helpers tree but **not** byte-for-byte the same formatting.


In [ ]:
def print_lineage_from_revision(revision_id: str, max_depth: int = 6):
    seen = set()

    def walk(rid: str, indent: int = 0):
        if rid in seen or indent > max_depth * 2:
            return
        seen.add(rid)
        rev = client.get_revision(rid)
        prefix = "  " * (indent // 2)
        who = f"name={rev.name!r}"
        print(f"{prefix}- {who}  rev={rev.id}")
        for src in rev.sources or []:
            edge = src.relationship_identifier or "-"
            print(
                f"{prefix}  <- {src.resource_type} res={src.resource_id} rev={src.revision_id} rel={edge}"
            )
            if src.revision_id:
                walk(src.revision_id, indent + 2)

    walk(revision_id)


final_p, final_rev = find_product_by_filename(job2, "named_cells.json")
assert final_rev is not None

print("Lineage for Job 2's named_cells.json:\n")
print_lineage_from_revision(final_rev.id, max_depth=6)


## Verify in the UI

Same checklist as the `chaining_jobs` notebook: Models / Jobs / Resources; ids and revision ids should match the cells above.


## Optional · Archive the model

`archive_model` soft-archives the tutorial Model (lineage and data remain).


In [ ]:
client.archive_model(model.id)
print("Archived model", model.id)
